In [ ]:
"""
    A practical example for fine-tuning an open-weight LLM using LoRA with Hugging Face Transformers and PEFT

    We will use:
        - LLaMA-style workflow
        - LoRA (efficient fine-tuning)
        - Small instruction dataset
        - Single GPU setup
"""

In [ ]:
# Dependencies
# !pip install transformers datasets peft accelerate bitsandbytes trl
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import TrainingArguments                  # define the hyperparameters
from trl import SFTTrainer                                  # supervised fine-tuning trainer

In [ ]:
# Load Model
model_name = "mistralai/Mistral-7B-Instruct-v0.2"           # a 7-billion-parameter model
tokenizer = AutoTokenizer.from_pretrained(model_name)       # download and initialize the tokenizer associated with Mistral
model = AutoModelForCausalLM.from_pretrained(               # load the actual LLM into memory
    model_name,
    load_in_4bit=True,                                      # activate quantization -> compress weights from 16 bits to 4 bits to lower usage of VRAM
    device_map="auto"                                       # automatically handle memory allocation
)

In [ ]:
# Configure LoRA: LoRA trains only small adapter matrices instead of the full model
lora_config = LoraConfig(
    r=16,                                           # the size of the low-rank matrices injected into the model
    lora_alpha=32,                                  # scaling factor for the LoRA weights, normally set to double the value of r
    target_modules=["q_proj", "v_proj"],            # where to inject the adapters inside Mistral's transformer architecture, here the Query and Value projection matrices
    lora_dropout=0.05,                              # randomly deactivate 5% of the LoRA parameters to avoid overfitting
    bias="none",                                    # bias parameters will not be trained
    task_type="CAUSAL_LM"                           # training a Causal Language Model
)

model = get_peft_model(model, lora_config)          # wrapping the base model with the lora_config parameters
model.print_trainable_parameters()

In [ ]:
# Format Dataset
dataset = load_dataset("json", data_files="train.json")

def format_example(example):
    text = f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""
    return {"text": text}

dataset = dataset.map(format_example)

In [ ]:
# Train
training_args = TrainingArguments(      # hyperparameters
    output_dir="./results",             # training checkpoint saving directory
    per_device_train_batch_size=2,      # number of training examples processed simultaneously on a single GPU
    gradient_accumulation_steps=4,      # accumulate the gradients for 4 steps before performing a weight update -> effective batch size = 8
    learning_rate=2e-4,                 # learning rate
    num_train_epochs=3,                 # number of epochs
    logging_steps=10,                   # print training metrics every 10 steps
    save_steps=100,                     # save a model checkpoint every 100 steps
    fp16=True                           # enable Mixed Precision training using 16-bit floats instead of full 32-bit precision to speed up training and reduce memory usage
)

trainer = SFTTrainer(                   # initialize supervised fine-tuning trainer
    model=model,
    train_dataset=dataset["train"],     # DatasetDict
    dataset_text_field="text",          # the key/column in the dataset that contains the raw text strings to be trained on
    args=training_args                  # hyperparameter configuration settings
)

In [ ]:
# Start Training
trainer.train()                         # kick off the entire training process

In [ ]:
# Save Model
trainer.model.save_pretrained("fine_tuned_model")   # only save the tiny LoRA adapters instead of the massive 15 GB file containing all 7 billion parameters
tokenizer.save_pretrained("fine_tuned_model")

In [ ]:
# Load Base and Fine-tuned Models
"""
    Since we only saved the adapters to keep things lightweight, we can't just load this folder by itself. 
    When we want to use the fine-tuned model, we need to load the base Mistral model first, then layer the saved adapters on top of it.
"""
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Load the original base model
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2", 
    load_in_4bit=True, 
    device_map="auto"
)
base_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

# 2. Load custom tokenizer and layer that saved LoRA adapters on top
ft_model = PeftModel.from_pretrained(base_model, "fine_tuned_model")
ft_tokenizer = AutoTokenizer.from_pretrained("fine_tuned_model")

# Now 'ft_model' is ready to generate text using fine-tuned knowledge

In [ ]:
# Generate Answers
def generate_answer(model, tokenizer, question):
    inputs = tokenizer(question, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

question = "What are the final standings for Germany's group in the 2026 World Cup? Which teams are advancing to the knockout stage?"

base_answer = generate_answer(
    base_model,
    base_tokenizer,
    question
)

ft_answer = generate_answer(
    ft_model,
    ft_tokenizer,
    question
)

print(">>> QUESTION: ", question)
print(">>> BASE LLM: ", base_answer)
print(">>> FINE-TUNED LLM: ", ft_answer)

In [ ]:
# Or Use Fine-tuned Model via Pipeline for Inference
from transformers import pipeline

pipe = pipeline(
    "text-generation",                              # specify the NLP task
    model="fine_tuned_model",                       # point to the folder where weights are saved
    tokenizer=ft_tokenizer                          # load tokenizer
)

result = pipe("What are the final standings for Germany's group in the 2026 World Cup? Which teams are advancing to the knockout stage?")
"""
    Behind the scenes: 
        The pipeline automatically takes this string, runs it through the tokenizer, 
        feeds those token IDs into the fine-tuned model, runs the model's generation loop, 
        collects the output token IDs, and decodes them back into human-readable text.
"""

# The result variable is returned as a list containing a dictionary
print(result[0]["generated_text"])